In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

entity ------> care_Epi_contract_id

MPB-------------

In [ ]:
-- Populate care_epi_contr_id using the unique contract ID from the new RDM Contracts table
,CAST(rdmc.contr_id AS varchar(100)) as care_epi_contr_id

In [ ]:
-- Join MPB tenancy contract to the new RDM Contracts table using source system instance and source contract ID
LEFT JOIN
    silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'MPB001'
   AND rdmc.contr_src_id = CONCAT('MPB ', CAST(ten.id AS varchar(100)))

In [ ]:
SELECT
    u.id,
    ten.id as tenancy_id,
    CONCAT('MPB ', CAST(ten.id AS varchar(100))) as expected_src_id,
    rdmc.contr_src_id,
    rdmc.contr_src_sys_inst_id,
    rdmc.contr_id as new_care_epi_contr_id
FROM (SELECT * FROM silver_drj_users WHERE profile_type = 'user') u
LEFT JOIN silver_drj_tenancies ten
    ON ten.id = u.tenancy_id
LEFT JOIN silver_rdm_contract rdmc
    ON rdmc.contr_src_sys_inst_id = 'MPB001'
   AND rdmc.contr_src_id = CONCAT('MPB ', CAST(ten.id AS varchar(100)))
LIMIT 50;

In [ ]:
Updated MPB care episode contract mapping to use the new RDM Contracts join via silver_rdm_contract, matching on contr_src_sys_inst_id and contr_src_id, and populating care_epi_contr_id from rdmc.contr_id instead of the old source-derived contract key.